In [1]:
from mip import *
import pandas as pd
from icecream import ic
import numpy as np

import io
from functools import reduce
import re
import math
from dataclasses import dataclass
from typing import Literal
from collections import defaultdict

In [2]:
pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

In [3]:
recipes_df = pd.read_excel('datasheet.ods', sheet_name='配方', index_col=1)

recipes_df = recipes_df[~recipes_df['禁用']].drop(columns=['禁用'])

is_powergen = recipes_df['工艺'].str.startswith('热能池')
powergen_df = recipes_df[is_powergen].reset_index()
is_metastorage = recipes_df['工艺'].str.startswith('超库存传输')
metastorage_df = recipes_df[is_metastorage].reset_index()
recipes_df = recipes_df[~(is_powergen | is_metastorage)].reset_index()

display(recipes_df, powergen_df, metastorage_df)

,功率,工艺,原料1,原料1每分钟消耗,原料2,原料2每分钟消耗,产物,分钟产率
0,-10,水泵,NaN,NaN,NaN,NaN,水,60.0
1,-5,精炼炉,蓝铁矿,-30.0,NaN,NaN,蓝铁块,30.0
2,-5,精炼炉,紫晶矿,-30.0,NaN,NaN,紫晶纤维,30.0
3,-5,精炼炉,源矿,-30.0,NaN,NaN,晶体外壳,30.0
4,-5,精炼炉,致密晶体粉末,-30.0,NaN,NaN,密制晶体,30.0
5,-5,精炼炉,致密蓝铁粉末,-30.0,NaN,NaN,钢块,30.0
6,-5,精炼炉,高晶粉末,-30.0,NaN,NaN,高晶纤维,30.0
7,-5,精炼炉,致密碳粉末,-30.0,NaN,NaN,稳定碳块,30.0
8,-5,精炼炉,致密源石粉末,-30.0,NaN,NaN,致密晶体粉末,30.0
9,-5,精炼炉,砂叶,-30.0,NaN,NaN,碳块,30.0


,功率,工艺,原料1,原料1每分钟消耗,原料2,原料2每分钟消耗,产物,分钟产率
0,50,热能池-源矿,源矿,-7.5,NaN,NaN,NaN,NaN
1,220,热能池-低容谷地电池,低容谷地电池,-1.5,NaN,NaN,NaN,NaN
2,420,热能池-中容谷地电池,中容谷地电池,-1.5,NaN,NaN,NaN,NaN
3,1100,热能池-高容谷地电池,高容谷地电池,-1.5,NaN,NaN,NaN,NaN
4,1600,热能池-低容武陵电池,低容武陵电池,-1.5,NaN,NaN,NaN,NaN


,功率,工艺,原料1,原料1每分钟消耗,原料2,原料2每分钟消耗,产物,分钟产率
0,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.016667,NaN,NaN,蓝铁矿,0.016667
1,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.016667,NaN,NaN,紫晶矿,0.016667
2,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.016667,NaN,NaN,源矿,0.016667
3,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.083333,NaN,NaN,钢质瓶,0.016667
4,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.033333,NaN,NaN,蓝铁瓶,0.016667
5,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.033333,NaN,NaN,紫晶质瓶,0.016667
6,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.033333,NaN,NaN,钢块,0.016667
7,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.033333,NaN,NaN,高晶纤维,0.016667
8,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.033333,NaN,NaN,密制晶体,0.016667
9,0,超库存传输-四号谷地,超库存传输值-四号谷地,-0.033333,NaN,NaN,致密蓝铁粉末,0.016667


In [4]:
materials = reduce(
    set.union,
    (
        set(recipes_df[col])
        for col in recipes_df.columns
        if re.match(r'^(原料|产物)\d?$', col)
    )
) - {np.nan, math.nan}
materials = list(materials)

_=ic(materials)

ic| materials: ['紫晶纤维',
                '高晶粉末',
                '高晶零件',
                '芽针溶液',
                '芽针粉末',
                '源石粉末',
                '钢质瓶',
                '钢块',
                '碳粉末',
                '蓝铁块',
                '源矿粉末',
                '芽针',
                '密制晶体',
                '紫晶粉末',
                '碳块',
                '钢制零件',
                '致密源石粉末',
                '低容谷地电池',
                '砂叶粉末',
                '低容武陵电池',
                '晶体粉末',
                '高容谷地电池',
                '砂叶',
                '晶体外壳',
                '蓝铁粉末',
                '稳定碳块',
                '紫晶质瓶',
                '铁制零件',
                '息壤',
                '中容谷地电池',
                '蓝铁矿',
                '高晶质瓶',
                '紫晶矿',
                '高晶纤维',
                '源矿',
                '紫晶零件',
                '致密碳粉末',
                '蓝铁瓶',
                '致密蓝铁粉末',
                '蓝铁瓶-芽针溶液',
                '水',
                '芽针针

In [5]:
INIT_POWER = 200.0
EXTERN_POWER = -700.0  # 基地外消耗
# INIT_POWER = np.inf # 手动搬电池
INIT_MATERIAL = {
    '源矿': 350.0,
    '紫晶矿': 0.0,
    '蓝铁矿': 90.0,
}
METASTORAGE_QUOTA_MIN = 1500.0 / 60.0
XIRANG_OVEN_LIMIT = 2
VALUES = {
    '低容武陵电池': 25.0,
    '芽针针剂': 16.0,
}
TARGET_VALUE = 279.5
MERGABLE_MACHINE = [
    # '精炼炉',
    # '粉碎机',
    # '配件机',
    # '塑形机',
    # '双种植采种循环',
    # '双种植采种循环-液体',
    # '单种植采种循环',
    # '单种植采种循环-液体',
    # '装备原件机',
    # '灌装机',
    # '封装机',
    # '研磨机',
    # '反应池',
    # '天有洪炉',
    # '拆解机',
]

n_recipes = recipes_df.shape[0]
n_powergen = powergen_df.shape[0]
n_metastorage = metastorage_df.shape[0]

m = Model(sense=MAXIMIZE)
x_recipes = m.add_var_tensor((n_recipes, ), 'x_recipes', lb=0.0, ub=INF, var_type=CONTINUOUS)
x_recipes_ceil = m.add_var_tensor((n_recipes, ), 'x_recipes_ceil', lb=0.0, ub=INF, var_type=INTEGER)
m += (x_recipes_ceil >= x_recipes)
x_powergen = m.add_var_tensor((n_powergen, ), 'x_powergen', lb=0.0, ub=INF, var_type=CONTINUOUS)
x_metastorage = m.add_var_tensor((n_metastorage, ), 'x_metastorage', lb=0.0, ub=1.0, var_type=INTEGER)

tot_power = INIT_POWER + EXTERN_POWER + xsum(
    x_recipes_ceil * recipes_df['功率']
) + xsum(
    x_powergen * powergen_df['功率']
)
m += (tot_power >= 0, 'tot_power')

tot_material = defaultdict(lambda: 0)
for mat, cnt in INIT_MATERIAL.items():
    tot_material[mat] += cnt
for i, row in enumerate(recipes_df.itertuples()):
    mat1, mat2, prd = getattr(row, '原料1'), getattr(row, '原料2'), getattr(row, '产物')
    x_i = x_recipes[i]
    if mat1 not in (np.nan, math.nan):
        tot_material[mat1] += x_i * getattr(row, '原料1每分钟消耗')
    if mat2 not in (np.nan, math.nan):
        tot_material[mat2] += x_i * getattr(row, '原料2每分钟消耗')
    if prd not in (np.nan, math.nan):
        tot_material[prd] += x_i * getattr(row, '分钟产率')
for i, row in enumerate(powergen_df.itertuples()):
    mat1, mat2, prd = getattr(row, '原料1'), getattr(row, '原料2'), getattr(row, '产物')
    x_i = x_powergen[i]
    if mat1 not in (np.nan, math.nan):
        tot_material[mat1] += x_i * getattr(row, '原料1每分钟消耗')
    if mat2 not in (np.nan, math.nan):
        tot_material[mat2] += x_i * getattr(row, '原料2每分钟消耗')
    if prd not in (np.nan, math.nan):
        tot_material[prd] += x_i * getattr(row, '分钟产率')
for i, row in enumerate(metastorage_df.itertuples()):
    prd = getattr(row, '产物')
    x_i = x_metastorage[i]
    tot_material[prd] += x_i * (METASTORAGE_QUOTA_MIN / -getattr(row, '原料1每分钟消耗') * getattr(row, '分钟产率'))
for mat, cnt in tot_material.items():
    m += (cnt >= 0, mat)

tot_xirang_oven = xsum(
    x_recipes_ceil[
        list(recipes_df[recipes_df['工艺'] == '天有洪炉'].index)
    ]
)
m += (tot_xirang_oven <= XIRANG_OVEN_LIMIT, '天有洪炉')

m += (xsum(x_metastorage) <= 1, '超库存传输')

tot_value = xsum(tot_material[mat] * val for mat, val in VALUES.items())
# m.objective = tot_value
m += tot_value >= TARGET_VALUE

m.objective = tot_material['息壤装备原件']

In [6]:
m.optimize(max_seconds=10)

Welcome to the CBC MILP Solver 
Version: Trunk
Build Date: Oct 24 2021 

Starting solution of the Linear programming relaxation problem using Dual Simplex

Coin0506I Presolve 62 (-29) rows, 84 (-35) columns and 210 (-87) elements
Clp0014I Perturbing problem by 0.001% of 15.578721 - largest nonzero change 0.00014111259 ( 0.00090580342%) - largest zero change 0.00013156542
Clp0000I Optimal - objective value 1.6899709
Coin0511I After Postsolve, objective 1.6899709, infeasibilities - dual 0 (0), primal 0 (0)
Clp0032I Optimal objective 1.689970886 - 59 iterations time 0.002, Presolve 0.00

Starting MIP optimization


<OptimizationStatus.OPTIMAL: 0>

In [7]:
ic(m.num_solutions)
# print('调度券分钟产量', m.objective_value)
print('调度券分钟产量', tot_value.x, '日产量', tot_value.x * 60 * 24, '周产量', tot_value.x * 60 * 24 * 7)
print('装备原件分钟产量', m.objective_value, '日产量', m.objective_value * 60 * 24, '周产量', m.objective_value * 60 * 24 * 7)
result_tot_power_machine = np.sum(np.array([u.x for u in x_recipes_ceil]) * recipes_df['功率'])
print('产线机器耗电', result_tot_power_machine)
print('最大允许耗电', result_tot_power_machine + EXTERN_POWER)

print('超库存传输', metastorage_df.loc[(u.x == 1.0 for u in x_metastorage)]['产物'].item())

result_df =pd.concat([
    pd.concat([
        recipes_df,
        pd.DataFrame(
            {'速率': u.x, '建造数量': v.x}
            for u, v in zip(x_recipes, x_recipes_ceil)
        ),
    ], axis=1),
    pd.concat([
        powergen_df,
        pd.DataFrame(
            {'速率': u.x, '建造数量': math.ceil(u)}
            for u in x_powergen
        ),
    ], axis=1)
])
result_df = result_df[result_df['建造数量'] != 0]
display(result_df)


None

调度券分钟产量 279.5 日产量 402480.0 周产量 2817360.0
装备原件分钟产量 1.6491071428571438 日产量 2374.714285714287 周产量 16623.00000000001
产线机器耗电 -1155.0
最大允许耗电 -1855.0
超库存传输 铁制零件


ic| m.num_solutions: 1


,功率,工艺,原料1,原料1每分钟消耗,原料2,原料2每分钟消耗,产物,分钟产率,速率,建造数量
0,-10,水泵,NaN,NaN,NaN,NaN,水,60.0,3.000000,3.0
1,-5,精炼炉,蓝铁矿,-30.0,NaN,NaN,蓝铁块,30.0,3.000000,3.0
4,-5,精炼炉,致密晶体粉末,-30.0,NaN,NaN,密制晶体,30.0,0.549702,1.0
7,-5,精炼炉,致密碳粉末,-30.0,NaN,NaN,稳定碳块,30.0,4.000000,4.0
8,-5,精炼炉,致密源石粉末,-30.0,NaN,NaN,致密晶体粉末,30.0,0.549702,1.0
11,-5,精炼炉,芽针,-30.0,NaN,NaN,碳块,60.0,2.000000,2.0
14,-5,粉碎机,源矿,-30.0,NaN,NaN,源石粉末,30.0,9.801190,10.0
15,-5,粉碎机,碳块,-30.0,NaN,NaN,碳粉末,60.0,4.000000,4.0
17,-5,粉碎机,砂叶,-30.0,NaN,NaN,砂叶粉末,90.0,3.000000,3.0
18,-5,粉碎机,芽针,-30.0,NaN,NaN,芽针粉末,60.0,1.000000,1.0
